# Chapter 5 — Generating Training Data with DeepSeek

**Goal:** Use a large model (DeepSeek) as a *teacher* to generate high-quality labeled training data.

Chapters 2-4 used hand-crafted synthetic data. That works for tutorials but doesn't scale. In practice, you use a large, capable model to generate diverse, realistic examples — then train a small model on those examples. This is the first half of **knowledge distillation**.

**What we'll do:**
1. Generate classification examples (bug/feature/praise/question)
2. Generate structured extraction examples (name/company/role/date → JSON)
3. Validate outputs programmatically
4. Save the datasets for Chapter 6 training

## 5.1 Connect to DeepSeek

DeepSeek uses an OpenAI-compatible API. Set your key as an environment variable — never hardcode it.

In [ ]:
import os, json, re, time
from openai import OpenAI

# Set your key before running: os.environ["DEEPSEEK_API_KEY"] = "sk-..."
api_key = os.environ.get("DEEPSEEK_API_KEY")

if not api_key:
    raise RuntimeError("Set DEEPSEEK_API_KEY environment variable first")

client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

def ask_deepseek(messages, temperature=0.7, max_tokens=512):
    """Send a prompt to DeepSeek and return the text response."""
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# Quick connectivity test
reply = ask_deepseek(
    [{"role": "user", "content": "Reply with just the word: connected"}],
    temperature=0.0,
    max_tokens=10,
)
print(f"DeepSeek: {reply}")

DeepSeek: connected


## 5.2 Generate Classification Data

We ask DeepSeek to create diverse user messages and label them. The key is a **system prompt that defines the task precisely** — this is prompt engineering for the teacher.

We generate in batches of 10 to keep API calls efficient.

In [4]:
CLASSIFY_SYSTEM = """\
You are a data labeling assistant. Given a short user message, classify it into exactly one category.

Categories:
- bug: something is broken, not working, or producing errors
- feature_request: asking for new functionality that doesn't exist yet
- praise: compliment, positive feedback, or appreciation
- question: asking how something works, what a policy is, or where to find something

Rules:
- A "how do I fix X" that implies X is broken is a BUG, not a question
- A "do you support X" that asks about existing capability is a QUESTION
- A "please add X" is a FEATURE_REQUEST
- "Love the app" or "great job" is PRAISE

Return ONLY a JSON array of objects with "text" and "label" fields. No markdown, no explanation."""

classification_data = []

for batch_num in range(4):
    user_prompt = f"Generate 10 diverse user messages (mix of bug reports, feature requests, praise, and questions for a productivity app). Return as JSON array."

    raw = ask_deepseek([
        {"role": "system", "content": CLASSIFY_SYSTEM},
        {"role": "user", "content": user_prompt},
    ], temperature=0.8, max_tokens=1024)

    # Parse and validate
    try:
        batch = json.loads(raw)
        for item in batch:
            text = item.get("text", "").strip()
            label = item.get("label", "").strip().lower()
            valid_labels = {"bug", "feature_request", "praise", "question"}
            if text and label in valid_labels:
                classification_data.append({"text": text, "label": label})
        print(f"Batch {batch_num+1}: {len(batch)} generated, {len(classification_data)} total valid")
    except json.JSONDecodeError:
        print(f"Batch {batch_num+1}: JSON parse failed, retrying...")
        # Retry once with lower temperature
        raw = ask_deepseek([
            {"role": "system", "content": CLASSIFY_SYSTEM},
            {"role": "user", "content": user_prompt},
        ], temperature=0.3, max_tokens=1024)
        try:
            batch = json.loads(raw)
            for item in batch:
                text = item.get("text", "").strip()
                label = item.get("label", "").strip().lower()
                if text and label in {"bug", "feature_request", "praise", "question"}:
                    classification_data.append({"text": text, "label": label})
            print(f"Batch {batch_num+1} (retry): {len(batch)} generated")
        except json.JSONDecodeError:
            print(f"Batch {batch_num+1}: retry also failed, skipping")

    time.sleep(0.5)  # be kind to the API

print(f"\nTotal classification examples: {len(classification_data)}")
print(f"\nSamples:")
for ex in classification_data[:3]:
    print(f"  [{ex['label']}] {ex['text']}")

Batch 1: 10 generated, 10 total valid
Batch 2: 10 generated, 20 total valid
Batch 3: 10 generated, 30 total valid
Batch 4: 10 generated, 40 total valid

Total classification examples: 40

Samples:
  [bug] The app crashes every time I try to export my notes as PDF.
  [feature_request] Could you add a dark mode option? It would be easier on my eyes.
  [praise] This app has completely transformed how I organize my tasks. Amazing work!


## 5.3 Generate Extraction Data

We ask DeepSeek to create sentences about people's jobs and extract fields as JSON. The teacher must produce BOTH the sentence AND the structured output.

In [5]:
EXTRACT_SYSTEM = """\
You are a data generation assistant. Create realistic sentences about a person starting a job, and extract structured fields as JSON.

For each example, return a JSON object with:
- "sentence": a natural English sentence describing a person joining a company in a role
- "fields": { "name": ..., "company": ..., "role": ..., "start_date": ... }

Rules:
- Vary sentence structure: "X joined Y as Z in W", "In W, X became Z at Y", "Y hired X as Z in W", etc.
- Use realistic names, company names, and job titles
- start_date format: "Month Year" (e.g., "March 2022")

Return ONLY a JSON array of these objects. No markdown, no explanation."""

extraction_data = []

for batch_num in range(3):
    raw = ask_deepseek([
        {"role": "system", "content": EXTRACT_SYSTEM},
        {"role": "user", "content": f"Generate 10 diverse job-start sentences with extracted fields. Vary names, companies, roles, and dates."},
    ], temperature=0.8, max_tokens=1024)

    try:
        batch = json.loads(raw)
        for item in batch:
            sentence = item.get("sentence", "").strip()
            fields = item.get("fields", {})
            required = {"name", "company", "role", "start_date"}
            if sentence and all(k in fields and fields[k] for k in required):
                extraction_data.append({"sentence": sentence, "fields": fields})
        print(f"Batch {batch_num+1}: {len(batch)} generated, {len(extraction_data)} total valid")
    except json.JSONDecodeError:
        print(f"Batch {batch_num+1}: JSON parse failed, retrying...")
        raw = ask_deepseek([
            {"role": "system", "content": EXTRACT_SYSTEM},
            {"role": "user", "content": f"Generate 10 diverse job-start sentences. Return ONLY a JSON array."},
        ], temperature=0.3, max_tokens=1024)
        try:
            batch = json.loads(raw)
            for item in batch:
                sentence = item.get("sentence", "").strip()
                fields = item.get("fields", {})
                if sentence and all(k in fields and fields[k] for k in {"name", "company", "role", "start_date"}):
                    extraction_data.append({"sentence": sentence, "fields": fields})
            print(f"Batch {batch_num+1} (retry): {len(batch)} generated")
        except json.JSONDecodeError:
            print(f"Batch {batch_num+1}: retry also failed, skipping")

    time.sleep(0.5)

print(f"\nTotal extraction examples: {len(extraction_data)}")
print(f"\nSamples:")
for ex in extraction_data[:3]:
    print(f"  {ex['sentence']}")
    print(f"  → {json.dumps(ex['fields'])}")

Batch 1: 10 generated, 10 total valid
Batch 2: 10 generated, 20 total valid
Batch 3: 10 generated, 30 total valid

Total extraction examples: 30

Samples:
  Emily Chen joined Google as a Software Engineer in March 2022.
  → {"name": "Emily Chen", "company": "Google", "role": "Software Engineer", "start_date": "March 2022"}
  In July 2021, James Rodriguez became a Data Analyst at Microsoft.
  → {"name": "James Rodriguez", "company": "Microsoft", "role": "Data Analyst", "start_date": "July 2021"}
  Amazon hired Sarah Kim as a Product Manager in November 2020.
  → {"name": "Sarah Kim", "company": "Amazon", "role": "Product Manager", "start_date": "November 2020"}


## 5.4 Validate & Deduplicate

Teacher models can produce near-duplicates or malformed outputs. Always validate before training.

In [6]:
# Check class balance
from collections import Counter
label_counts = Counter(ex["label"] for ex in classification_data)
print("Classification — class distribution:")
for label, count in label_counts.most_common():
    print(f"  {label:<20} {count}")

# Remove near-duplicates (simple substring check)
unique_class = []
seen_texts = set()
for ex in classification_data:
    normalized = ex["text"].lower().strip()
    if normalized not in seen_texts:
        seen_texts.add(normalized)
        unique_class.append(ex)

dup_count = len(classification_data) - len(unique_class)
print(f"\nDuplicates removed: {dup_count}")
classification_data = unique_class

# Check extraction data for duplicate sentences
unique_extract = []
seen_sentences = set()
for ex in extraction_data:
    norm = ex["sentence"].lower().strip()
    if norm not in seen_sentences:
        seen_sentences.add(norm)
        unique_extract.append(ex)
print(f"Extraction duplicates removed: {len(extraction_data) - len(unique_extract)}")
extraction_data = unique_extract

print(f"\nFinal:")
print(f"  Classification: {len(classification_data)} examples")
print(f"  Extraction:     {len(extraction_data)} examples")

Classification — class distribution:
  bug                  12
  feature_request      12
  praise               8
  question             8

Duplicates removed: 0
Extraction duplicates removed: 0

Final:
  Classification: 40 examples
  Extraction:     30 examples


## 5.5 Save the Datasets

Save as JSON so Chapter 6 can load them directly for fine-tuning and distillation.

In [7]:
os.makedirs("./generated", exist_ok=True)

with open("./generated/classification.json", "w") as f:
    json.dump(classification_data, f, indent=2)

with open("./generated/extraction.json", "w") as f:
    json.dump(extraction_data, f, indent=2)

print("Saved:")
print(f"  generated/classification.json ({len(classification_data)} items)")
print(f"  generated/extraction.json ({len(extraction_data)} items)")

Saved:
  generated/classification.json (40 items)
  generated/extraction.json (30 items)


## 5.6 What We Just Did — and Why

| Concept | What it means |
|---|---|
| **Teacher model** | A large, capable model (DeepSeek) that generates training data |
| **Student model** | A small model (SmolLM) that will be trained on the teacher's outputs |
| **System prompt** | Defines the labeling rules — this is the *curriculum* the teacher follows |
| **Temperature** | Higher (0.8) for diversity, lower (0.3) for reliability on retry |
| **Output validation** | Parsing, field checks, dedup — teacher models make mistakes too |

**The key insight:** Instead of hand-writing 400 examples (like we did in Chapters 2-4), we prompted a large model to generate them in seconds. The quality is higher, the diversity is greater, and the approach scales to thousands of examples.

In Chapter 6 we'll use this data to train SmolLM and compare it against the hand-crafted data approach — then extend to full knowledge distillation with soft labels.